In [1]:
import os
import json
import pickle
import re
import pprint
from dataclasses import dataclass
from typing import List, Dict, Tuple

import torch
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from FeatureMapping import ROOM_CODE, FEATURES_LIST, ROOM_NAME_BY_CODE, FeatureMapping


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
COLOR_MAPPING = {
    (0xFF, 0xD7, 0x00): 'common_room',
    (0xFF, 0xA5, 0x00): 'master_room',
    (0xEE, 0xE8, 0xAA): 'living_room',
    (0x6B, 0x8E, 0x23): 'balcony',
    (0xAD, 0xD8, 0xE6): 'bathroom',
    (0xF0, 0x80, 0x80): 'kitchen',
    (0xDD, 0xA0, 0xDD): 'storage',
    (0xDA, 0x70, 0xD6): 'dining',
}

In [5]:
class LocalLLM:
    def __init__(self, model_id: str, device: str = 'cuda'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map='auto',
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        self.device = self.model.device

    def __call__(self, prompt: str, **generate_kwargs) -> str:
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.device)

        defaults = dict(max_new_tokens=256, do_sample=False)
        params = {**defaults, **generate_kwargs}
        with torch.no_grad():
            out = self.model.generate(**inputs, **params)
        text = self.tokenizer.decode(out[0], skip_special_tokens=True)

        return text[len(prompt):].strip()

LLM_CACHE: Dict[str, LocalLLM] = {}

MODEL_MAP = {
    'qwen2.5': 'Qwen/Qwen2.5-7B-Instruct',
    'llama3.1': 'meta-llama/Llama-3.1-8B-Instruct',
    'gemma2': 'google/gemma-2-9b-it'
}

def get_llm_api(model_name: str):
    """Return an async function that sends `prompt` to the specified local LLM."""
    if model_name not in MODEL_MAP:
        raise ValueError(f"Unknown model: {model_name}")
    if model_name not in LLM_CACHE:
        LLM_CACHE[model_name] = LocalLLM(MODEL_MAP[model_name])
    return LLM_CACHE[model_name]

In [6]:
class LLMPromptGenerator:
    def __init__(self, feature_mapping: FeatureMapping):
        self.mapping = feature_mapping

    def generate_image_prompt(self,
                              sample_files: List[str],
                              num_shots: int) -> str:
       
        color_legend = ", ".join([
            f"{room}: RGB{color}" for color, room in COLOR_MAPPING.items()
        ])
        prompt = (
            f"You are an expert architect analyzing apartment floor plans. "
            f"You have {num_shots} example plans provided. Rooms are color-coded as follows: {color_legend}.\n"
            f"Floor plan filenames: {', '.join(sample_files)}\n"
            "Identify common spatial patterns, adjacency relationships, and proportions. "
            "Be specific with approximate measurements or ratios when possible."
        )
        return prompt

    def generate_description_prompt(self,
                                    descriptions: List[str],
                                    num_shots: int) -> str:
        
        prompt = (
            f"Analyze these {num_shots} verbal descriptions of apartments with similar layouts. "
            "Identify common room types, spatial relationships, size patterns, and distinguishing features.\n\n"
            "Descriptions:\n"
        )
        for i, desc in enumerate(descriptions, 1):
            prompt += f"Example {i}: {desc}\n"
        prompt += (
            "\nProvide a concise summary of typical room arrangements, proportions, and unique characteristics."
        )
        return prompt

    def generate_graph_prompt(self,
                              graphs: List[Dict],
                              num_shots: int) -> str:
        
        # sample_json = json.dumps(graphs, indent=2)
        # prompt = (
        #     f"Analyze these {num_shots} apartment graph representations (nodes=rooms, edges=adjacencies). ")
        # #  f"Graphs JSON:\n{sample_json}\n"
        # for i, desc in enumerate(graphs, 1):
        #     prompt += f"Example {i}: {desc}\n"
        # prompt += (
        #     "Identify common node types, adjacency pairs, and spatial distributions. "
        #     "Provide quantitative summaries where applicable."
        # )
        prompt = (
            f"Analyze these {num_shots} apartment graph representations. Note that  nodes=rooms, edges=adjacencies."
            "Note that these representations also include graphs where nodes=rooms, edges=adjacencies. "
            "Identify common room types, spatial relationships, size patterns, and distinguishing features.\n\n"
            "Graph Examples:\n"
        )
        for i, graph in enumerate(graphs, 1):
            example_str = pprint.pformat(graph, indent=2, width=80)
            prompt += f"Example {i}:\n{example_str}\n\n"
        prompt += "Provide a concise summary of typical room arrangements, proportions, and unique characteristics."
        return prompt

In [ ]:
# class LLMResponseAnalyzer:
#     def __init__(self, feature_mapping: FeatureMapping):
#         self.mapping = feature_mapping
#         self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

#     def extract_features(self, response: str) -> Dict:
#         extracted = {
#             'spatial': [], 'size': [], 'adjacency': [], 'shape': [], 'sequence': [],
#             'quantitative': {'numbers': [], 'ratios': [], 'percentages': []}
#         }
#         # Quantitative
#         extracted['quantitative']['numbers'] = re.findall(r"\b\d+\.?\d*\b", response)
#         extracted['quantitative']['ratios'] = re.findall(r"\b\d+\.?\d*:\d+\.?\d*\b", response)
#         extracted['quantitative']['percentages'] = re.findall(r"\b\d+\.?\d*%\b", response)
#         # simple keyword-based extracts
#         for kw in ['adjacent','next to','connected','between','central','corner']:
#             if kw in response.lower(): extracted['spatial'].append(kw)
#         for kw in ['large','small','area','size','square','rectangular','elongated','narrow']:
#             if kw in response.lower():
#                 extracted['size' if kw in ['large','small','area','size','square'] else 'shape'].append(kw)
#         return extracted

#     def map_to_features(self, llm_obs: Dict) -> List[str]:

#         contexts = llm_obs['spatial'] + llm_obs['size'] + llm_obs['adjacency'] + llm_obs['shape']
#         if not contexts: return []
#         emb_ctx = self.embedder.encode(contexts)
#         keys, vals = zip(*self.mapping.clustering_to_concept.items())
#         emb_feats = self.embedder.encode(list(vals))
#         sims = cosine_similarity(emb_ctx, emb_feats)
#         mapped = set()
#         for i in range(sims.shape[0]):
#             for j,score in enumerate(sims[i]):
#                 if score > 0.5:
#                     mapped.add(keys[j])
#         return list(mapped)

In [ ]:
# class ClusteringLLMComparator:

#     def __init__(self,
#                  cluster_profiles: Dict[int,List[str]],
#                  top_features: Dict[int,List[str]]):
#         self.cluster_profiles = cluster_profiles
#         self.top_features = top_features

#     def compare(self, cluster_id: int, mapped: List[str]) -> Dict:
#         cf = set(self.top_features.get(cluster_id, []))
#         ll = set(mapped)
#         overlap = cf & ll
#         prec = len(overlap)/len(ll) if ll else 0
#         rec = len(overlap)/len(cf) if cf else 0
#         f1  = 2*prec*rec/(prec+rec) if prec+rec>0 else 0
#         return {
#             'cluster': cluster_id,
#             'precision': prec,
#             'recall': rec,
#             'f1': f1,
#             'overlap': list(overlap),
#             'clustering_only': list(cf-ll),
#             'llm_only': list(ll-cf)
#         }

In [7]:
cluster_csv = "./examples/spectral/cluster_results.csv"
fi_csv = "./examples/spectral/cluster_features.csv"

annotation_dir = "./annotation/human_annotated_tags"

merged = pd.read_csv(cluster_csv)
cluster_profiles = merged.groupby('label')[FEATURES_LIST].mean().to_dict(orient='index')
# Load top features per cluster
df_fi = pd.read_csv(fi_csv)
top_feats = {
    cid: grp.sort_values('importance',ascending=False)['feature'].tolist()
    for cid, grp in df_fi.groupby('cluster_id')
}


list_of_files = [f.split('.')[0] for f in os.listdir(annotation_dir)]
files_df = pd.DataFrame(list_of_files, columns=['id'])
merged['id'] = merged['filename'].str.split(".").str[0]

merged = merged.merge(files_df, how="inner", on="id").reset_index(drop=True).drop(columns=["id"])
merged['filename'] = merged['filename'].str.split(".").str[0]

graph_pkl = "./output/graphs_reoriented_dict.pkl"
# Load graphs
with open(graph_pkl, 'rb') as f:
    graphs = pickle.load(f)


In [8]:
descriptions = {}
for fn in merged['filename']:
    path = os.path.join(annotation_dir, fn + '.txt')
    with open(path, 'r') as f: 
        descriptions[fn] = f.read().strip()

In [9]:

mapping        = FeatureMapping()
prompt_gen     = LLMPromptGenerator(mapping)
# resp_analyzer  = LLMResponseAnalyzer(mapping)
# comparator     = ClusteringLLMComparator(cluster_profiles, top_feats)


In [10]:
models = ['llama3.1']
# models = ['qwen2.5']
# models = ['gemma2']
shot_counts = [5, 10]

In [ ]:
results = {}  

for model in models:
    llm_api = get_llm_api(model)  # now a sync callable returning str
    results[model] = {}
    for shots in shot_counts:
        results[model][shots] = {}
        for cluster_id, group in merged.groupby('label'):
            samples = group['filename'].tolist()[:shots]
            print(samples)

            # # 1) description prompt & response
            # descs    = [descriptions[f] for f in samples]
            # prompt_d = prompt_gen.generate_description_prompt(descs, shots)
            # resp_d   = llm_api(prompt_d)

            # 2) graph prompt & response
            grs      = [graphs[f] for f in samples]
            prompt_g = prompt_gen.generate_graph_prompt(grs, shots)
            resp_g   = llm_api(prompt_g)

            # store only raw text
            results[model][shots][cluster_id] = {
                # 'description': resp_d,
                'graph': resp_g
            }

with open('raw_llama_json_responses.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [28]:
import os
import shutil
import json
shots = 15
# Configuration
base_output = './examples/spectral'
human_anno_dir = './annotation/human_annotated_tags'
floorplan_dir = './data/floorplan_reoriented'
# graphs is assumed to be a dict mapping filename (without extension) to a graph object
# with a method or property that can be converted to a dict/json
# e.g., graphs['123'] -> {'nodes': [...], 'edges': [...]}

# Ensure base output directory exists
os.makedirs(base_output, exist_ok=True)

for cluster_id, group in merged.groupby('label'):
    # take up to `shots` samples from this cluster
    samples = group['filename'].tolist()[:shots]

    # Set up cluster folder
    cluster_folder = os.path.join(base_output, f'cluster_{cluster_id}')
    human_out = os.path.join(cluster_folder, 'human_annotation')
    json_out = os.path.join(cluster_folder, 'json')
    floorplan_out = os.path.join(cluster_folder, 'floorplan_reoriented')

    # Create subdirectories
    for d in (human_out, json_out, floorplan_out):
        os.makedirs(d, exist_ok=True)

    # Process each sample
    for fname in samples:
 
        src_txt = os.path.join(human_anno_dir, f'{fname}.txt')
        dst_txt = os.path.join(human_out, f'{fname}.txt')
        if os.path.exists(src_txt):
            shutil.copy(src_txt, dst_txt)
        else:
            print(f'annotation file not found: {src_txt}')


        graph = graphs.get(fname)
        if graph is not None:
            graph_data = graph if isinstance(graph, dict) else graph.to_dict()
            pretty_txt = pprint.pformat(graph_data, indent=2)
            dst_pretty = os.path.join(json_out, f'{fname}.txt')
            with open(dst_pretty, 'w') as pf:
                pf.write(pretty_txt)
        else:
            print(f'graph missing for {fname}')

        # Copy floorplan image
        src_img = os.path.join(floorplan_dir, f'{fname}.png')
        dst_img = os.path.join(floorplan_out, f'{fname}.png')
        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
        else:
            print(f'floorplan image not found: {src_img}')


